# 02 — Engenharia de features

Duas passadas, conforme combinado:

**Passada 1** (por ativo): retornos/lags → vol. realizada → Garman-Klass → GARCH(1,1)
expansivo (refit mensal, sem lookahead) → skew/curtose móveis → spread de
Corwin-Schultz → vol. implícita (VIX + Ibov/Bloomberg). Salva
`data/processed/features.parquet`.

**Passada 2** (painel *pooled*, todos os ativos juntos): ADF/KPSS → correlação →
filtro VIF (>10) → relatório de features mantidas/descartadas — um único
conjunto de features para todos os ativos (arquitetura simétrica do LSTM).

`NaN` estrutural (burn-in do GARCH, `implied_vol_br` pré-2015, gaps de dados
> 3 dias úteis) é mantido em `features.parquet` — o corte final da amostra de
modelagem fica para quando o dataset de treino for montado (Semana 5-6).

In [1]:
import sys
from pathlib import Path

import pandas as pd
import yaml

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.features import align_and_fill_ohlc, build_asset_features
from src.data.feature_selection import correlation_matrix, stationarity_report, vif_filter

config = yaml.safe_load((PROJECT_ROOT / "config.yaml").read_text(encoding="utf-8"))
end = config["sample"]["end_date"]

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLES_DIR = PROJECT_ROOT / "reports" / "tables"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

## Montar o painel: início comum entre os 7 ativos e tratamento de missing

Soja/café usam os futuros Yahoo ajustados por rolagem (decisão registrada em
`PLANO.md` > Pendências — CEPEA fica só para robustez futura). O início comum
do painel é definido pela equity/FX mais tardia (as commodities cobrem 2005
inteiro, então nunca são elas que limitam).

In [2]:
equities_fx = pd.read_parquet(RAW_DIR / "equities_fx.parquet")
futures_adjusted = pd.read_parquet(RAW_DIR / "futures_adjusted_yahoo.parquet")
implied_vol = pd.read_parquet(RAW_DIR / "implied_vol.parquet")

equity_fx_tickers = [a["ticker"] for a in config["assets"] if a["category"] in ("equity", "fx")]
futures_tickers = [a["ticker"] for a in config["assets"] if a["category"] == "commodity"]

core_ohlc = {t: equities_fx[t] for t in equity_fx_tickers}
core_ohlc.update({t: futures_adjusted[t] for t in futures_tickers})

common_start = max(df["Close"].dropna().index.min() for df in core_ohlc.values())
print("início comum do painel:", common_start.date())

início comum do painel: 2007-06-29


In [3]:
aligned_ohlc = {ticker: align_and_fill_ohlc(df, common_start, end) for ticker, df in core_ohlc.items()}
implied_vol_aligned = implied_vol.reindex(pd.bdate_range(common_start, end))

## Passada 1 — features por ativo

In [4]:
features_by_asset = {}
for ticker, ohlc in aligned_ohlc.items():
    asset_features = build_asset_features(
        ohlc,
        lags=tuple(config["features"]["returns_lags"]),
        windows=tuple(config["features"]["rolling_windows"]),
    )
    features_by_asset[ticker] = pd.concat([asset_features, implied_vol_aligned], axis=1)

features_by_asset["SLCE3.SA"].tail()

,return_1d,return_5d,return_21d,realized_vol_21d,realized_vol_63d,garman_klass_vol_21d,garman_klass_vol_63d,garch_cond_vol,skew_21d,kurtosis_21d,skew_63d,kurtosis_63d,corwin_schultz_spread,volume_normalized,implied_vol_us,implied_vol_br
2024-12-25,-0.003927,-0.047746,0.050140,0.276680,0.226830,0.296280,0.260303,0.018285,0.255037,0.238995,0.279114,0.719536,0.018161,0.807558,NaN,NaN
2024-12-26,-0.003943,-0.038743,0.037371,0.276472,0.221549,0.291489,0.254112,0.018361,0.365979,0.302807,0.312156,0.905741,0.013988,0.695272,14.73,15.4928
2024-12-27,-0.002260,-0.040460,0.009091,0.262218,0.220369,0.276122,0.253841,0.018436,0.524451,1.043687,0.354775,1.006703,0.007221,0.983944,15.95,15.2315
2024-12-30,-0.010233,-0.024274,0.005156,0.263840,0.220632,0.269938,0.254564,0.018510,0.541287,0.971211,0.406346,1.020945,0.000000,0.736532,17.40,14.8137
2024-12-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
features_panel = pd.concat(features_by_asset, axis=1)
features_panel.to_parquet(PROCESSED_DIR / "features.parquet")
print("features.parquet salvo:", features_panel.shape)

features.parquet salvo: (4568, 112)


## Passada 2 — diagnóstico no painel *pooled* (ADF/KPSS, correlação, VIF)

In [6]:
pooled = pd.concat(features_by_asset.values(), axis=0)
pooled_complete = pooled.dropna()
print(
    "painel pooled para diagnóstico:", pooled_complete.shape,
    "(de", pooled.shape[0], "linhas totais antes do dropna — reflete o NaN estrutural do burn-in)",
)

stationarity = stationarity_report(pooled_complete)
corr = correlation_matrix(pooled_complete)
kept_features, vif_report = vif_filter(pooled_complete, threshold=config["features"]["vif_threshold"])

stationarity

painel pooled para diagnóstico: (13818, 16) (de 31976 linhas totais antes do dropna — reflete o NaN estrutural do burn-in)


,adf_stat,adf_pvalue,adf_rejects_unit_root,kpss_stat,kpss_pvalue,kpss_fails_to_reject_stationary,stationary_both_tests
feature,,,,,,,
return_1d,-20.961275,0.000000e+00,True,0.132305,0.100000,True,True
return_5d,-20.092658,0.000000e+00,True,0.152243,0.100000,True,True
return_21d,-14.015269,3.652459e-26,True,0.190758,0.100000,True,True
realized_vol_21d,-9.588085,2.071180e-16,True,2.506162,0.010000,False,False
realized_vol_63d,-8.689943,4.066263e-14,True,2.582682,0.010000,False,False
garman_klass_vol_21d,-7.377554,8.642174e-11,True,6.497269,0.010000,False,False
garman_klass_vol_63d,-8.086212,1.417594e-12,True,6.281008,0.010000,False,False
garch_cond_vol,-9.474148,4.031651e-16,True,6.270707,0.010000,False,False
skew_21d,-18.166723,2.453119e-30,True,0.861008,0.010000,False,False


In [7]:
vif_report

,status,vif
feature,,
garman_klass_vol_63d,dropped,12.319915
return_1d,kept,1.262777
return_5d,kept,1.577733
return_21d,kept,1.417222
realized_vol_21d,kept,4.561034
realized_vol_63d,kept,3.833100
garman_klass_vol_21d,kept,4.556960
garch_cond_vol,kept,3.729677
skew_21d,kept,1.414857


In [8]:
stationarity.to_csv(TABLES_DIR / "feature_stationarity.csv")
corr.to_csv(TABLES_DIR / "feature_correlation.csv")
vif_report.to_csv(TABLES_DIR / "feature_vif_selection.csv")

print(f"Features retidas após VIF (<= {config['features']['vif_threshold']}):")
print(kept_features)
print()
print("Features descartadas por multicolinearidade:")
print(vif_report.index[vif_report["status"] == "dropped"].tolist())

Features retidas após VIF (<= 10):
['return_1d', 'return_5d', 'return_21d', 'realized_vol_21d', 'realized_vol_63d', 'garman_klass_vol_21d', 'garch_cond_vol', 'skew_21d', 'kurtosis_21d', 'skew_63d', 'kurtosis_63d', 'corwin_schultz_spread', 'volume_normalized', 'implied_vol_us', 'implied_vol_br']

Features descartadas por multicolinearidade:
['garman_klass_vol_63d']
